# Nepali News Headline/Article Classifier

**Goal:** Build a model that reads a Nepali news headline/article and predicts its category (sports, politics, entertainment, etc.)

**Plan:**
1. Get the dataset (iNLTK Nepali News Dataset, 5 categories)
2. Preprocess Nepali text
3. Baseline model: TF-IDF + Logistic Regression
4. Evaluate + error analysis
5. (Later notebook) Fine-tune a pretrained Nepali BERT model

Run this in Google Colab — go to colab.research.google.com, upload this notebook (File > Upload notebook).

## Step 1: Get the dataset

You need a free Kaggle account and API token to download programmatically:

1. Go to kaggle.com > your profile > Settings > API > "Create New Token" — this downloads a file called `kaggle.json`
2. Run the cell below, and when prompted, upload that `kaggle.json` file
3. It will then download and unzip the dataset automatically

Dataset used: `disisbig/nepali-news-dataset` (8,000+ articles, 5 categories)

In [16]:
import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

!pip install -q kaggle
!rm -rf nepali_news_data nepali-news-dataset.zip
!kaggle datasets download -d disisbig/nepali-news-dataset
!unzip -q nepali-news-dataset.zip -d nepali_news_data
!echo "Done. Contents:"
!ls -R nepali_news_data | head -50


Dataset URL: https://www.kaggle.com/datasets/disisbig/nepali-news-dataset
License(s): CC-BY-SA-4.0
100% 1.08M/1.08M [00:01<00:00, 1.08MB/s]

Done. Contents:
nepali_news_data:
train.csv
valid.csv
Dataset URL: https://www.kaggle.com/datasets/disisbig/nepali-news-dataset
License(s): CC-BY-SA-4.0
100% 1.08M/1.08M [00:01<00:00, 1.11MB/s]

Done. Contents:
nepali_news_data:
train.csv
valid.csv


In [17]:
results_df = pd.DataFrame({
    'text': X_test.values,
    'actual': y_test.values,
    'predicted': bert_preds_labels
})
misclassified = results_df[results_df['actual'] != results_df['predicted']]
print(f"{len(misclassified)} of {len(results_df)} test examples misclassified")
misclassified.sample(min(10, len(misclassified)), random_state=1)


24 of 1494 test examples misclassified


,text,actual,predicted
516,धुलाम्मे र अस्तव्यस्त विराटनगरको मूल सडकमा माल...,entertainment,business
815,अदालतले सन् २०१५ को मार्चमा पार्मा एफसीलाई टाट...,sports,entertainment
198,विश्वभरका हिमाल आरोहण गर्दै हिँड्ने दावा याङ्ज...,entertainment,business
608,एपीएफ र मनाङ मर्स्याङ्दीको अन्तिम सेमिफाइनल खे...,entertainment,sports
1048,उमेरले ७० कटेका पूर्णबहादुर विश्वकर्मा ७ वर्षी...,entertainment,business
772,चितवन राष्ट्रिय निकुञ्जका तत्कालीन सहायक वार्ड...,entertainment,business
460,नेपाल च्याम्बर अफ कमर्स हङकङ एनसीसीएचके र पोखर...,business,sports
263,कर्णालीको कृषि पर्यटन र इतिहाससहित विविध पक्ष ...,business,entertainment
147,गायक आनन्द कार्की र गीतकार रोशन भट्टराईलाई पोख...,entertainment,business
957,अमेरिकी टेलिभिजन स्टार काइली जेनर खर्बपतिहरूको...,entertainment,business


## Step 2: Inspect the data

This dataset is typically organized as folders per category, each containing text files.
The cell below auto-detects the structure — if it doesn't match, print `nepali_news_data`'s
contents above and adjust the loading code accordingly. Datasets on Kaggle sometimes
change format, so treat this as a starting point, not gospel.

In [14]:
bert_preds_raw = trainer.predict(test_ds)
bert_preds = bert_preds_raw.predictions.argmax(axis=1)
bert_preds_labels = le.inverse_transform(bert_preds)
print(classification_report(y_test, bert_preds_labels))

               precision    recall  f1-score   support

     business       0.98      0.99      0.98       525
entertainment       0.98      0.97      0.98       454
       sports       0.99      0.99      0.99       515

     accuracy                           0.98      1494
    macro avg       0.98      0.98      0.98      1494
 weighted avg       0.98      0.98      0.98      1494



In [9]:
import pandas as pd

train_df = pd.read_csv('nepali_news_data/train.csv')
valid_df = pd.read_csv('nepali_news_data/valid.csv')

# Combine into one df with 'text' and 'label' columns
# Using 'paras' (article body) as text since it has more content than just the heading
train_df = train_df.rename(columns={'paras': 'text'})[['text', 'label']]
valid_df = valid_df.rename(columns={'paras': 'text'})[['text', 'label']]

df = pd.concat([train_df, valid_df], ignore_index=True)

print(f"Loaded {len(df)} documents")
print(df['label'].value_counts())
df.head()

Loaded 7470 documents
label
business         2628
sports           2574
entertainment    2268
Name: count, dtype: int64


,text,label
0,नेपाली कथानक फिल्म ‘लभ स्टेसन’ को टिम यति बेला...,entertainment
1,दसैंको मुखमा अस्वाभाविक बढेको तरकारी तथा फलफूल...,business
2,एशियाकै ठूलो बियर कम्पनी मध्येको युनाइटेड ब्रु...,business
3,संसारका धनाढ्यहरू अन्तरिक्ष यात्रालाई सस्तो र ...,business
4,निकेश खड्का निर्देशित फिल्म ‘फाटेको जुत्ता’ को...,entertainment


## Step 3: Preprocess Nepali text

Key differences from English preprocessing:
- **Unicode normalization** (NFC) — Devanagari can encode the same visible character multiple byte-ways; skipping this causes silent bugs where "identical" words don't match
- Strip everything except Devanagari script + whitespace
- Whitespace tokenization is imperfect for Nepali (postpositions/compounds attach), but it's a fine starting baseline

In [10]:
import re
import unicodedata

def preprocess_nepali(text):
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'[^\u0900-\u097F\s]', '', text)  # keep only Devanagari + whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(preprocess_nepali)

# Drop rows that became empty after cleaning
before = len(df)
df = df[df['clean_text'].str.len() > 0].reset_index(drop=True)
print(f"Dropped {before - len(df)} empty rows after cleaning")
df[['label', 'clean_text']].head()

Dropped 0 empty rows after cleaning


,label,clean_text
0,entertainment,नेपाली कथानक फिल्म लभ स्टेसन को टिम यति बेला भ...
1,business,दसैंको मुखमा अस्वाभाविक बढेको तरकारी तथा फलफूल...
2,business,एशियाकै ठूलो बियर कम्पनी मध्येको युनाइटेड ब्रु...
3,business,संसारका धनाढ्यहरू अन्तरिक्ष यात्रालाई सस्तो र ...
4,entertainment,निकेश खड्का निर्देशित फिल्म फाटेको जुत्ता को ह...


## Step 4: Baseline model — TF-IDF + Logistic Regression

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, stratify=df['label'], random_state=42
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_vec, y_train)

preds = clf.predict(X_test_vec)
print(classification_report(y_test, preds))

               precision    recall  f1-score   support

     business       0.91      0.91      0.91       525
entertainment       0.88      0.92      0.90       454
       sports       0.96      0.92      0.94       515

     accuracy                           0.92      1494
    macro avg       0.92      0.92      0.92      1494
 weighted avg       0.92      0.92      0.92      1494



## Step 5: Error analysis

This is the part that makes it look like real work rather than a tutorial copy-paste.
Look at the confusion matrix, find which categories get mixed up, and pull a few
actual misclassified examples to read and reason about *why*.

In [12]:
import pandas as pd

labels_sorted = sorted(df['label'].unique())
cm = confusion_matrix(y_test, preds, labels=labels_sorted)
cm_df = pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted)
print("Confusion matrix (rows=actual, cols=predicted):")
cm_df

Confusion matrix (rows=actual, cols=predicted):


,business,entertainment,sports
business,478,36,11
entertainment,28,419,7
sports,21,20,474


In [13]:
# Pull a handful of misclassified examples to actually read
results_df = pd.DataFrame({
    'text': X_test.values,
    'actual': y_test.values,
    'predicted': preds
})
misclassified = results_df[results_df['actual'] != results_df['predicted']]
print(f"{len(misclassified)} of {len(results_df)} test examples misclassified\n")
misclassified.sample(min(10, len(misclassified)), random_state=1)

123 of 1494 test examples misclassified



,text,actual,predicted
515,०७५ को पहिलो चौमासिक सकिन एक साता मात्रै बाँकी...,entertainment,business
1351,प्रादेशिक संरचनाअनुसार अनुगमनको जिम्मेवारी स्थ...,business,entertainment
878,प्रतियोगिताको व्यवस्थापनबारे पछि भन्छु भनेको थ...,sports,entertainment
1219,झापाको तापक्रम करिब ४० डिग्री नाघ्ने तरखरमा थि...,entertainment,business
488,उनी खेतका गह्रामा काठको ब्याट बनाएर क्रिकेट खे...,sports,entertainment
325,राष्ट्रपति डोनाल्ड ट्रम्पले अमेरिका र चीन व्या...,business,entertainment
791,यति धेरै खर्च फेरि उही पुरानै कथा । एक अर्को व...,sports,entertainment
491,टिन्ज क्रयु सेन्टर धनकुटाद्वारा आयोजित पूर्वाञ...,entertainment,business
1146,गत वर्ष शत्रु गते मार्फत फिल्म निर्माणमा फर्कि...,entertainment,business
957,अमेरिकी टेलिभिजन स्टार काइली जेनर खर्बपतिहरूको...,entertainment,business


## Next steps (not in this notebook yet)

- Fine-tune a pretrained Nepali BERT model (e.g. search HuggingFace for current Nepali checkpoints — availability changes, so verify the model card before using) and compare accuracy against this baseline
- Try the harder 16-category dataset (`sndsabin/Nepali-News-Classifier` on GitHub) once this pipeline is solid
- Write up a short README comparing baseline vs. fine-tuned results — this is the portfolio piece

In [7]:
!pip install -q transformers datasets accelerate

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder

# Encode string labels to integers (business/entertainment/sports -> 0/1/2)
le = LabelEncoder()
y_train_int = le.fit_transform(y_train)
y_test_int = le.transform(y_test)
num_labels = len(le.classes_)
print("Label mapping:", dict(zip(le.classes_, range(num_labels))))

model_name = "Shushant/nepaliBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

def tokenize_fn(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

train_ds = Dataset.from_dict({'text': list(X_train), 'label': list(y_train_int)}).map(tokenize_fn, batched=True)
test_ds = Dataset.from_dict({'text': list(X_test), 'label': list(y_test_int)}).map(tokenize_fn, batched=True)

args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",
    save_strategy="no",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds)
trainer.train()

Label mapping: {'business': 0, 'entertainment': 1, 'sports': 2}


config.json:   0%|          | 0.00/589 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/529k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Map:   0%|          | 0/1494 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,0.078315,0.058590
2,0.032794,0.069130
3,0.010652,0.076466


TrainOutput(global_step=1122, training_loss=0.0633022849399882, metrics={'train_runtime': 417.1899, 'train_samples_per_second': 42.973, 'train_steps_per_second': 2.689, 'total_flos': 1179274338256896.0, 'train_loss': 0.0633022849399882, 'epoch': 3.0})